In [11]:
import Bio
import pandas as pd
import numpy as np
import seaborn as sns
import os
import logomaker
import matplotlib.pyplot as plt
import scipy.stats as sp
from scipy.stats import ttest_ind
from scipy.stats import t
plt.rcParams['savefig.dpi'] = 300
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Data import IUPACData

# COMPILING Q/QQ ENDINGS

In this set of cells, I am parsing through the fasta file of the entire yeast proteome to find proteins that end in Q, QQ or QQQ, as well as all proteins that end in duplicate amino acids generally. I used the file: uniprotkb_proteome_UP000002311_2024_09_20.fasta, downloaded from https://www.uniprot.org/uniprotkb?query=proteome:UP000002311.

In [12]:
records = list(SeqIO.parse("uniprotkb_proteome_UP000002311_2024_09_20.fasta", "fasta"))
#obtained from https://www.uniprot.org/uniprotkb?query=proteome:UP000002311
len(records)

6060

In [3]:
print(records[1:3])
#checking file to see how sequence is annotated

[SeqRecord(seq=Seq('MSDAAPSLSNLFYDPTYNPGQSTINYTSIYGNGSTITFDELQGLVNSTVTQAIM...NNL'), id='sp|D6VTK4|STE2_YEAST', name='sp|D6VTK4|STE2_YEAST', description='sp|D6VTK4|STE2_YEAST Pheromone alpha factor receptor OS=Saccharomyces cerevisiae (strain ATCC 204508 / S288c) OX=559292 GN=STE2 PE=1 SV=2', dbxrefs=[]), SeqRecord(seq=Seq('MSYTDNPPQTKRALSLDDLVNHDENEKVKLQKLSEAANGSRPFAENLESDINQT...KVM'), id='sp|O13297|CET1_YEAST', name='sp|O13297|CET1_YEAST', description='sp|O13297|CET1_YEAST mRNA-capping enzyme subunit beta OS=Saccharomyces cerevisiae (strain ATCC 204508 / S288c) OX=559292 GN=CET1 PE=1 SV=2', dbxrefs=[])]


In [4]:
endsinQ = [record for record in records if str(record.seq).endswith('Q')]
endsinQQ = [record for record in records if str(record.seq).endswith('QQ')]
endsinQQQ = [record for record in records if str(record.seq).endswith('QQQ')]

In [5]:
endsinQQQ
#only one yeast protein ends in QQQ (PUB1) so I opted not to save that one 

[SeqRecord(seq=Seq('MSENNEEQHQQQQQQQPVAVETPSAVEAPASADPSSEQSVAVEGNSEQAEDNQG...QQQ'), id='sp|P32588|PUB1_YEAST', name='sp|P32588|PUB1_YEAST', description='sp|P32588|PUB1_YEAST Nuclear and cytoplasmic polyadenylated RNA-binding protein PUB1 OS=Saccharomyces cerevisiae (strain ATCC 204508 / S288c) OX=559292 GN=PUB1 PE=1 SV=4', dbxrefs=[])]

In [6]:
#saving Q endings
Q_endings = {
    "id": [record.id for record in endsinQ],
    "name": [record.name for record in endsinQ],
    "description": [record.description for record in endsinQ],
    "sequence": [str(record.seq) for record in endsinQ]
}
df = pd.DataFrame(Q_endings)
df

df.to_csv('filtered_seqrecords_endsinQ.csv', index=False)

In [8]:
#saving QQ endings
QQ_endings = {
    "id": [record.id for record in endsinQQ],
    "name": [record.name for record in endsinQQ],
    "description": [record.description for record in endsinQQ],
    "sequence": [str(record.seq) for record in endsinQQ]
}
df = pd.DataFrame(QQ_endings)
df

df.to_csv('filtered_seqrecords_endsinQQ.csv', index=False)

# COMPILING ALL DUPLICATE AA ENDINGS

In [17]:
# Get all amino acids
amino_acids = list(IUPACData.protein_letters)  # Standard one-letter codes for amino acids

# Generate all duplicates and triplicates
duplicates = [aa * 2 for aa in amino_acids]
triplicates = [aa * 3 for aa in amino_acids]

# Function to filter by suffix
def filter_by_suffix(records, suffix):
    """Filter sequences that end with a specific suffix."""
    return [record for record in records if str(record.seq).endswith(suffix)]

# Filter for duplicates
filtered_duplicates = {suffix: filter_by_suffix(records, suffix) for suffix in duplicates}

# Filter for triplicates
filtered_triplicates = {suffix: filter_by_suffix(records, suffix) for suffix in triplicates}

In [18]:
filtered_duplicates

{'AA': [SeqRecord(seq=Seq('MARYGATSTNPAKSASARGSYLRVSFKNTRETAQAINGWELTKAQKYLEQVLDH...IAA'), id='sp|P05740|RL17A_YEAST', name='sp|P05740|RL17A_YEAST', description='sp|P05740|RL17A_YEAST Large ribosomal subunit protein uL22A OS=Saccharomyces cerevisiae (strain ATCC 204508 / S288c) OX=559292 GN=RPL17A PE=1 SV=4', dbxrefs=[]),
  SeqRecord(seq=Seq('MTNEKVWIEKLDNPTLSVLPHDFLRPQQEPYTKQATYSLQLPQLDVPHDSFSNK...AAA'), id='sp|P07702|LYS2_YEAST', name='sp|P07702|LYS2_YEAST', description='sp|P07702|LYS2_YEAST L-2-aminoadipate reductase OS=Saccharomyces cerevisiae (strain ATCC 204508 / S288c) OX=559292 GN=LYS2 PE=1 SV=2', dbxrefs=[]),
  SeqRecord(seq=Seq('MDISKPVGSEITSVDFGILTAKEIRNLSAKQITNPTVLDNLGHPVSGGLYDLAL...NAA'), id='sp|P10964|RPA1_YEAST', name='sp|P10964|RPA1_YEAST', description='sp|P10964|RPA1_YEAST DNA-directed RNA polymerase I subunit RPA190 OS=Saccharomyces cerevisiae (strain ATCC 204508 / S288c) OX=559292 GN=RPA190 PE=1 SV=2', dbxrefs=[]),
  SeqRecord(seq=Seq('MSATSTATSTSASQLHLNSTPVTHCLSDIVK

In [20]:
#saving duplicate endings to dataframe and csv
ZZ_endings = {
    "id": [],
    "name": [],
    "description": [],
    "sequence": [],
    "duplicate_identity": []  
}

for amino_acid, seq_list in filtered_duplicates.items():
    for record in seq_list:
        ZZ_endings["id"].append(record.id)
        ZZ_endings["name"].append(record.name)
        ZZ_endings["description"].append(record.description)
        ZZ_endings["sequence"].append(str(record.seq))
        ZZ_endings["duplicate_identity"].append(amino_acid)  

for key, value in ZZ_endings.items():
    print(f"{key}: {len(value)}") 

df = pd.DataFrame(ZZ_endings)

df.to_csv('filtered_seqrecords_duplicates.csv', index=False)

print(df)

id: 560
name: 560
description: 560
sequence: 560
duplicate_identity: 560
                        id                   name  \
0    sp|P05740|RL17A_YEAST  sp|P05740|RL17A_YEAST   
1     sp|P07702|LYS2_YEAST   sp|P07702|LYS2_YEAST   
2     sp|P10964|RPA1_YEAST   sp|P10964|RPA1_YEAST   
3     sp|P32318|THI4_YEAST   sp|P32318|THI4_YEAST   
4    sp|P32472|FKBP2_YEAST  sp|P32472|FKBP2_YEAST   
..                     ...                    ...   
555   sp|Q12116|TMS1_YEAST   sp|Q12116|TMS1_YEAST   
556  sp|P0C5M9|YF009_YEAST  sp|P0C5M9|YF009_YEAST   
557   sp|P36127|YK12_YEAST   sp|P36127|YK12_YEAST   
558  sp|Q3E7Z1|YN146_YEAST  sp|Q3E7Z1|YN146_YEAST   
559  sp|Q3E7Z7|YD003_YEAST  sp|Q3E7Z7|YD003_YEAST   

                                           description  \
0    sp|P05740|RL17A_YEAST Large ribosomal subunit ...   
1    sp|P07702|LYS2_YEAST L-2-aminoadipate reductas...   
2    sp|P10964|RPA1_YEAST DNA-directed RNA polymera...   
3    sp|P32318|THI4_YEAST Thiamine thiazole synthas...   


# Making master sequence file of +/-9 nucleotides surrounding stop codon for every gene in the yeast genome

In [24]:
#load fasta and gtf files
genome_seq = SeqIO.to_dict(SeqIO.parse("Saccharomyces_cerevisiae.R64-1-1.dna.toplevel.fa", "fasta"))
in_file = "stop_codon_output.gtf"
#I have already filtered the gtf file from the original download by running this in command line:
#awk '$3 == "stop_codon"' Saccharomyces_cerevisiae.R64-1-1.110.gtf > stop_codon_output.gtf 

In [25]:
#load GFFexaminer & examine gtf file
import pprint
from BCBio.GFF import GFFExaminer

examiner = GFFExaminer()
in_handle = open(in_file)
pprint.pprint(examiner.available_limits(in_handle))
in_handle.close()

{'gff_id': {('I',): 117,
            ('II',): 456,
            ('III',): 184,
            ('IV',): 836,
            ('IX',): 241,
            ('Mito',): 28,
            ('V',): 323,
            ('VI',): 139,
            ('VII',): 583,
            ('VIII',): 321,
            ('X',): 398,
            ('XI',): 348,
            ('XII',): 578,
            ('XIII',): 505,
            ('XIV',): 435,
            ('XV',): 597,
            ('XVI',): 511},
 'gff_source': {('sgd',): 6600},
 'gff_source_type': {('sgd', 'stop_codon'): 6600},
 'gff_type': {('stop_codon',): 6600}}


In [37]:
#This code is for filtering through the fasta file using the gtf file 
from BCBio import GFF
from Bio import SeqIO
import csv

# Read in the sequence file
in_seq_file = "Saccharomyces_cerevisiae.R64-1-1.dna.toplevel.fa"
in_seq_handle = open(in_seq_file)
seq_dict = SeqIO.to_dict(SeqIO.parse(in_seq_handle, "fasta"))
in_seq_handle.close()

# Read in the expanded stop codon GTF file and extract ranges
ranges_data = []

# Parse the GFF/GTG file
in_handle = open(in_file)
for rec in GFF.parse(in_handle, base_dict=seq_dict):
    for feature in rec.features:
        if feature.type == "stop_codon":  #ensure we're only looking at stop codons 
            #(.gtf should be exclusively stop codons anyway)
            #this is the start and end of the stop codon
            original_start = feature.location.start
            original_end = feature.location.end
            
            # Expand range by 9 nucleotides on either side (total of 18)
            expanded_start = max(0, original_start - 9)  #avoid negatives
            expanded_end = original_end + 9 
            
            chromosome = rec.id
            gene_id = feature.qualifiers.get("gene_name", ["unknown"])[0]
            transcript_id = feature.qualifiers.get("transcript_id", ["unknown"])[0]
            strand = "+" if feature.location.strand == 1 else "-"  
            
            sequence = str(seq_dict[chromosome].seq[expanded_start:expanded_end])
            
            ranges_data.append([chromosome, gene_id, transcript_id, strand, expanded_start, expanded_end, sequence])

in_handle.close()

# Write to CSV file with sequences included
with open("stop_codon_ranges_with_sequences_yeast.csv", "w", newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow(["Chromosome", "Gene_name","Transcript_ID", "Strand", "Start", "End", "Sequence"])
    csv_writer.writerows(ranges_data)

print("Data saved to stop_codon_ranges_with_sequences_yeast.csv")

Data saved to stop_codon_ranges_with_sequences_yeast.csv


In [42]:
df_all = pd.read_csv('stop_codon_ranges_with_sequences_yeast.csv')
#cool, but negative strands will display reverse complemented sequences
#need to change that
df_all

,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence
0,I,unknown,YAL037W_mRNA,+,74811,74832,GGTCGCTTTTGAAGAGCCCTC
1,I,TPD3,YAL016W_mRNA,+,126774,126795,TTAAAAAATTGATACTAGTTC
2,I,EFB1,YAL003W_mRNA,+,143148,143169,CAAAAATTATAAAAGGCTTTT
3,I,FLC2,YAL053W_mRNA,+,48238,48259,CCATATTTGTAATCCAATATA
4,I,unknown,YAL031W-A_mRNA,+,84965,84986,ATTGGCGGATAATGTAGCACA
...,...,...,...,...,...,...,...
6595,XVI,HDA3,YPR179C_mRNA,-,893787,893808,TTTATTTAATTATACATTTCC
6596,XVI,unknown,YPR096C_mRNA,-,724832,724853,CAAAAGAATTCAAAGTACTAT
6597,XVI,ERG10,YPL028W_mRNA,+,499280,499301,GAAAAGATATGATTACGTTCT
6598,XVI,NIP100,YPL174C_mRNA,-,220157,220178,ATCACTCAGTCATTTGTAATA


In [77]:
def reverse_complement(seq):
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    return ''.join(complement[base] for base in reversed(seq))

#reverse complement sequence based on strand
def replace_with_reverse_complement(row):
    if row['Strand']=="-":
        row['Sequence'] = reverse_complement(row['Sequence'])
    return row

df_all_new = df_all.apply(replace_with_reverse_complement, axis=1)

#drop _mRNA, add new columns for stop codon, +4 nucleotide and 3UTR codon
df_all_new['Transcript_ID'] = df_all_new['Transcript_ID'].str.replace('_mRNA', '')
df_all_new['Stop Codon'] = df_all_new['Sequence'].str[9:12]
df_all_new['+4 nucleotide'] = df_all_new['Sequence'].str[12:13]
df_all_new['3UTR codon'] = df_all_new['Sequence'].str[12:15]

#save the resulting DataFrame (this now contains all strands in the 5 to 3 direction)
df_all_new.to_csv('whole_genome_stop_ranges_060326.csv', index=False)

df_all_new

,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon
0,I,unknown,YAL037W,+,74811,74832,GGTCGCTTTTGAAGAGCCCTC,TGA,A,AGA
1,I,TPD3,YAL016W,+,126774,126795,TTAAAAAATTGATACTAGTTC,TGA,T,TAC
2,I,EFB1,YAL003W,+,143148,143169,CAAAAATTATAAAAGGCTTTT,TAA,A,AAG
3,I,FLC2,YAL053W,+,48238,48259,CCATATTTGTAATCCAATATA,TAA,T,TCC
4,I,unknown,YAL031W-A,+,84965,84986,ATTGGCGGATAATGTAGCACA,TAA,T,TGT
...,...,...,...,...,...,...,...,...,...,...
6595,XVI,HDA3,YPR179C,-,893787,893808,GGAAATGTATAATTAAATAAA,TAA,T,TTA
6596,XVI,unknown,YPR096C,-,724832,724853,ATAGTACTTTGAATTCTTTTG,TGA,A,ATT
6597,XVI,ERG10,YPL028W,+,499280,499301,GAAAAGATATGATTACGTTCT,TGA,T,TTA
6598,XVI,NIP100,YPL174C,-,220157,220178,TATTACAAATGACTGAGTGAT,TGA,C,CTG


In [53]:
#loading gene names downloaded from https://www.alliancegenome.org/bluegenes/alliancemine/results/ALL_Yeast_Genes
#this is to match common gene name to systematic gene name since the proteome 
mapping_df = pd.read_csv('gene_names.csv')
mapping_df

,Gene > Primary DBID,Gene > Systematic Name,Gene > Organism . Short Name,Gene > Standard Name,Gene > Name
0,SGD:L000003336,YCLX01W,S. cerevisiae,NaN,NaN
1,SGD:S000000001,YAL001C,S. cerevisiae,TFC3,Transcription Factor class C
2,SGD:S000000002,YAL002W,S. cerevisiae,VPS8,Vacuolar Protein Sorting
3,SGD:S000000003,YAL003W,S. cerevisiae,EFB1,Elongation Factor Beta
4,SGD:S000000004,YAL005C,S. cerevisiae,SSA1,Stress-Seventy subfamily A
...,...,...,...,...,...
7168,SGD:S000350095,YFR035W-A,S. cerevisiae,NaN,NaN
7169,SGD:S000350096,YGR016C-A,S. cerevisiae,NaN,NaN
7170,SGD:S000350097,YMR106W-A,S. cerevisiae,NaN,NaN
7171,SGD:S000350098,YNL040C-A,S. cerevisiae,NaN,NaN


In [51]:
gene_mapping = dict(zip(mapping_df['Gene > Systematic Name'], mapping_df['Gene > Standard Name']))
gene_mapping

{'YCLX01W': nan,
 'YAL001C': 'TFC3',
 'YAL002W': 'VPS8',
 'YAL003W': 'EFB1',
 'YAL005C': 'SSA1',
 'YAL007C': 'ERP2',
 'YAL008W': 'FUN14',
 'YAL009W': 'SPO7',
 'YAL010C': 'MDM10',
 'YAL011W': 'SWC3',
 'YAL012W': 'CYS3',
 'YAL013W': 'DEP1',
 'YAL014C': 'SYN8',
 'YAL015C': 'NTG1',
 'YAL016W': 'TPD3',
 'YAL017W': 'PSK1',
 'YAL018C': 'LDS1',
 'YAL019W': 'FUN30',
 'YAL020C': 'ATS1',
 'YAL021C': 'CCR4',
 'YAL022C': 'FUN26',
 'YAL023C': 'PMT2',
 'YAL024C': 'LTE1',
 'YAL025C': 'MAK16',
 'YAL026C': 'DRS2',
 'YAL027W': 'SAW1',
 'YAL028W': 'FRT2',
 'YAL029C': 'MYO4',
 'YAL030W': 'SNC1',
 'YAL031C': 'GIP4',
 'YAL032C': 'PRP45',
 'YAL033W': 'POP5',
 'YAL034W-A': 'MTW1',
 'YAL035W': 'FUN12',
 'YAL036C': 'RBG1',
 'YAL037W': nan,
 'YAL038W': 'CDC19',
 'YAL039C': 'CYC3',
 'YAL040C': 'CLN3',
 'YAL041W': 'CDC24',
 'YAL042W': 'ERV46',
 'YAL043C': 'PTA1',
 'YAL044C': 'GCV3',
 'YAL045C': nan,
 'YAL046C': 'BOL3',
 'YAL047C': 'SPC72',
 'YAL048C': 'GEM1',
 'YAL049C': 'AIM2',
 'YAL051W': 'OAF1',
 'YAL053W': 'FLC

In [74]:
#correct gene names are in the description after GN=, including slashes
list_ZZ = dict(zip(df['description'].str.extract(r'GN=([\w-]+)')[0], df['duplicate_identity']))

In [192]:
#reverses order of dictionary (flip to common: systematic)
common_to_systematic = {v: k for k, v in gene_mapping.items()}

result = []
for gene, aa in list_ZZ.items():
    if gene in common_to_systematic:
        systematic = common_to_systematic[gene]
        common = gene
    elif gene in gene_mapping:
        systematic = gene
        common = np.nan
    else:
        systematic = np.nan
        common = np.nan
    result.append({'systematic_name': systematic, 'common_name': gene, 'amino_acid': aa})
    #captures genes (3) that didn't have a match in gene_mapping for common name 

df_result = pd.DataFrame(result)

In [193]:
merged = df_result.merge(df_all_new, left_on='systematic_name', right_on='Transcript_ID')
merged

,systematic_name,common_name,amino_acid,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon
0,YKL180W,RPL17A,AA,XI,RPL17A,YKL180W,+,110117,110138,ATTGCTGCTTAAATAGAAGAT,TAA,A,ATA
1,YBR115C,LYS2,AA,II,LYS2,YBR115C,-,469738,469759,GCAGCAGCTTAAGGTTGAGCA,TAA,G,GGT
2,YOR341W,RPA190,AA,XV,RPA190,YOR341W,+,965969,965990,AATGCGGCTTAGAAGCAGGCA,TAG,A,AAG
3,YGR144W,THI4,AA,VII,THI4,YGR144W,+,781367,781388,TTTGCTGCTTAGGCTTTTAAT,TAG,G,GCT
4,YDR519W,FPR2,AA,IV,FPR2,YDR519W,+,1480820,1480841,TCAGCCGCCTAGGTAAATTGG,TAG,G,GTA
...,...,...,...,...,...,...,...,...,...,...,...,...,...
552,YDR105C,TMS1,YY,IV,TMS1,YDR105C,-,665339,665360,AATTACTATTAAATAGACTAA,TAA,A,ATA
553,YFR009W-A,YFR009W-A,YY,VI,unknown,YFR009W-A,+,164120,164141,ACCTATTATTGAAAGATGTTA,TGA,A,AAG
554,YKR032W,YKR032W,YY,XI,unknown,YKR032W,+,506819,506840,TTGTATTACTGAGCGGGATAT,TGA,G,GCG
555,YNL146C-A,YNL146C-A,YY,XIV,unknown,YNL146C-A,-,351376,351397,TTATATTATTAAACCTATTTT,TAA,A,ACC


In [194]:
#these genes didn't have a match for their annotated common name
missing = df_result[~df_result['systematic_name'].isin(merged['systematic_name'])]
print(missing)
#look up manually and add systematic name

    systematic_name common_name amino_acid
12              NaN        BSC2         AA
488             NaN        MDN1         SS
504             NaN        MCP2         SS


In [195]:
new_rows = pd.DataFrame([
    {'common_name': 'BSC2', 'systematic_name': 'YDR275W', 'amino_acid' : 'AA'},
    {'common_name': 'MDN1', 'systematic_name': 'YLR106C', 'amino_acid' : 'SS'},
    {'common_name': 'MCP2', 'systematic_name': 'YLR253W', 'amino_acid' : 'SS'}
])

merged = pd.concat([merged, new_rows], ignore_index=True)
merged

,systematic_name,common_name,amino_acid,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon
0,YKL180W,RPL17A,AA,XI,RPL17A,YKL180W,+,110117.0,110138.0,ATTGCTGCTTAAATAGAAGAT,TAA,A,ATA
1,YBR115C,LYS2,AA,II,LYS2,YBR115C,-,469738.0,469759.0,GCAGCAGCTTAAGGTTGAGCA,TAA,G,GGT
2,YOR341W,RPA190,AA,XV,RPA190,YOR341W,+,965969.0,965990.0,AATGCGGCTTAGAAGCAGGCA,TAG,A,AAG
3,YGR144W,THI4,AA,VII,THI4,YGR144W,+,781367.0,781388.0,TTTGCTGCTTAGGCTTTTAAT,TAG,G,GCT
4,YDR519W,FPR2,AA,IV,FPR2,YDR519W,+,1480820.0,1480841.0,TCAGCCGCCTAGGTAAATTGG,TAG,G,GTA
...,...,...,...,...,...,...,...,...,...,...,...,...,...
555,YNL146C-A,YNL146C-A,YY,XIV,unknown,YNL146C-A,-,351376.0,351397.0,TTATATTATTAAACCTATTTT,TAA,A,ACC
556,YDR003W-A,YDR003W-A,YY,IV,unknown,YDR003W-A,+,454892.0,454913.0,ACATACTATTGATTTAGGTTT,TGA,T,TTT
557,YDR275W,BSC2,AA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
558,YLR106C,MDN1,SS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [196]:
#remove columns so I can merge again with three additional missing genes
merged_clean = merged[['systematic_name', 'common_name', 'amino_acid']]
merged_clean

,systematic_name,common_name,amino_acid
0,YKL180W,RPL17A,AA
1,YBR115C,LYS2,AA
2,YOR341W,RPA190,AA
3,YGR144W,THI4,AA
4,YDR519W,FPR2,AA
...,...,...,...
555,YNL146C-A,YNL146C-A,YY
556,YDR003W-A,YDR003W-A,YY
557,YDR275W,BSC2,AA
558,YLR106C,MDN1,SS


In [197]:
new_merged = merged_clean.merge(df_all_new, left_on='systematic_name', right_on='Transcript_ID', how='left')
new_merged

,systematic_name,common_name,amino_acid,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon
0,YKL180W,RPL17A,AA,XI,RPL17A,YKL180W,+,110117,110138,ATTGCTGCTTAAATAGAAGAT,TAA,A,ATA
1,YBR115C,LYS2,AA,II,LYS2,YBR115C,-,469738,469759,GCAGCAGCTTAAGGTTGAGCA,TAA,G,GGT
2,YOR341W,RPA190,AA,XV,RPA190,YOR341W,+,965969,965990,AATGCGGCTTAGAAGCAGGCA,TAG,A,AAG
3,YGR144W,THI4,AA,VII,THI4,YGR144W,+,781367,781388,TTTGCTGCTTAGGCTTTTAAT,TAG,G,GCT
4,YDR519W,FPR2,AA,IV,FPR2,YDR519W,+,1480820,1480841,TCAGCCGCCTAGGTAAATTGG,TAG,G,GTA
...,...,...,...,...,...,...,...,...,...,...,...,...,...
555,YNL146C-A,YNL146C-A,YY,XIV,unknown,YNL146C-A,-,351376,351397,TTATATTATTAAACCTATTTT,TAA,A,ACC
556,YDR003W-A,YDR003W-A,YY,IV,unknown,YDR003W-A,+,454892,454913,ACATACTATTGATTTAGGTTT,TGA,T,TTT
557,YDR275W,BSC2,AA,IV,BSC2,YDR275W,+,1012947,1012968,AAAGCTGCTTAAATTGATAAT,TAA,A,ATT
558,YLR106C,MDN1,SS,XII,MDN1,YLR106C,-,348996,349017,GCATCCAGCTAAGTATATCGT,TAA,G,GTA


In [198]:
#find mito genes
new_merged[new_merged['systematic_name'].str.startswith('Q')]

,systematic_name,common_name,amino_acid,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon
308,Q0297,Q0297,KK,Mito,unknown,Q0297,+,85697,85718,AAAAAAAAGTAATATATATAT,TAA,T,TAT
373,Q0120,BI4,NN,Mito,BI4,Q0120,+,42239,42260,AATAATAATTAAATTTTCTCC,TAA,A,ATT


In [199]:
new_merged = new_merged[~new_merged['systematic_name'].isin(['Q0297', 'Q0120'])] #drop mito genes

In [200]:
new_merged[new_merged['systematic_name'].str.startswith('Q')] #check if mito genes still there
new_merged

,systematic_name,common_name,amino_acid,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon
0,YKL180W,RPL17A,AA,XI,RPL17A,YKL180W,+,110117,110138,ATTGCTGCTTAAATAGAAGAT,TAA,A,ATA
1,YBR115C,LYS2,AA,II,LYS2,YBR115C,-,469738,469759,GCAGCAGCTTAAGGTTGAGCA,TAA,G,GGT
2,YOR341W,RPA190,AA,XV,RPA190,YOR341W,+,965969,965990,AATGCGGCTTAGAAGCAGGCA,TAG,A,AAG
3,YGR144W,THI4,AA,VII,THI4,YGR144W,+,781367,781388,TTTGCTGCTTAGGCTTTTAAT,TAG,G,GCT
4,YDR519W,FPR2,AA,IV,FPR2,YDR519W,+,1480820,1480841,TCAGCCGCCTAGGTAAATTGG,TAG,G,GTA
...,...,...,...,...,...,...,...,...,...,...,...,...,...
555,YNL146C-A,YNL146C-A,YY,XIV,unknown,YNL146C-A,-,351376,351397,TTATATTATTAAACCTATTTT,TAA,A,ACC
556,YDR003W-A,YDR003W-A,YY,IV,unknown,YDR003W-A,+,454892,454913,ACATACTATTGATTTAGGTTT,TGA,T,TTT
557,YDR275W,BSC2,AA,IV,BSC2,YDR275W,+,1012947,1012968,AAAGCTGCTTAAATTGATAAT,TAA,A,ATT
558,YLR106C,MDN1,SS,XII,MDN1,YLR106C,-,348996,349017,GCATCCAGCTAAGTATATCGT,TAA,G,GTA


In [201]:
new_merged = new_merged.drop(columns=['systematic_name', 'common_name'])
new_merged

,amino_acid,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon
0,AA,XI,RPL17A,YKL180W,+,110117,110138,ATTGCTGCTTAAATAGAAGAT,TAA,A,ATA
1,AA,II,LYS2,YBR115C,-,469738,469759,GCAGCAGCTTAAGGTTGAGCA,TAA,G,GGT
2,AA,XV,RPA190,YOR341W,+,965969,965990,AATGCGGCTTAGAAGCAGGCA,TAG,A,AAG
3,AA,VII,THI4,YGR144W,+,781367,781388,TTTGCTGCTTAGGCTTTTAAT,TAG,G,GCT
4,AA,IV,FPR2,YDR519W,+,1480820,1480841,TCAGCCGCCTAGGTAAATTGG,TAG,G,GTA
...,...,...,...,...,...,...,...,...,...,...,...
555,YY,XIV,unknown,YNL146C-A,-,351376,351397,TTATATTATTAAACCTATTTT,TAA,A,ACC
556,YY,IV,unknown,YDR003W-A,+,454892,454913,ACATACTATTGATTTAGGTTT,TGA,T,TTT
557,AA,IV,BSC2,YDR275W,+,1012947,1012968,AAAGCTGCTTAAATTGATAAT,TAA,A,ATT
558,SS,XII,MDN1,YLR106C,-,348996,349017,GCATCCAGCTAAGTATATCGT,TAA,G,GTA


In [202]:
new_merged['4-stop signal']=new_merged['Stop Codon'] + new_merged['+4 nucleotide']
new_merged

,amino_acid,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon,4-stop signal
0,AA,XI,RPL17A,YKL180W,+,110117,110138,ATTGCTGCTTAAATAGAAGAT,TAA,A,ATA,TAAA
1,AA,II,LYS2,YBR115C,-,469738,469759,GCAGCAGCTTAAGGTTGAGCA,TAA,G,GGT,TAAG
2,AA,XV,RPA190,YOR341W,+,965969,965990,AATGCGGCTTAGAAGCAGGCA,TAG,A,AAG,TAGA
3,AA,VII,THI4,YGR144W,+,781367,781388,TTTGCTGCTTAGGCTTTTAAT,TAG,G,GCT,TAGG
4,AA,IV,FPR2,YDR519W,+,1480820,1480841,TCAGCCGCCTAGGTAAATTGG,TAG,G,GTA,TAGG
...,...,...,...,...,...,...,...,...,...,...,...,...
555,YY,XIV,unknown,YNL146C-A,-,351376,351397,TTATATTATTAAACCTATTTT,TAA,A,ACC,TAAA
556,YY,IV,unknown,YDR003W-A,+,454892,454913,ACATACTATTGATTTAGGTTT,TGA,T,TTT,TGAT
557,AA,IV,BSC2,YDR275W,+,1012947,1012968,AAAGCTGCTTAAATTGATAAT,TAA,A,ATT,TAAA
558,SS,XII,MDN1,YLR106C,-,348996,349017,GCATCCAGCTAAGTATATCGT,TAA,G,GTA,TAAG


In [210]:
scores = pd.read_csv('stopcodonscores.csv') #load scoring file based on data from https://journals.plos.org/plosgenetics/article?id=10.1371/journal.pgen.1009538
score_dict = dict(zip(scores['Stop'], scores['Score'])) #saving scores to dictionary
new_merged['Score'] = new_merged['4-stop signal'].map(score_dict) #mapping scores to each 4-stop signal in dataset
new_merged

,amino_acid,Chromosome,Gene_name,Transcript_ID,Strand,Start,End,Sequence,Stop Codon,+4 nucleotide,3UTR codon,4-stop signal,Score
0,AA,XI,RPL17A,YKL180W,+,110117,110138,ATTGCTGCTTAAATAGAAGAT,TAA,A,ATA,TAAA,11
1,AA,II,LYS2,YBR115C,-,469738,469759,GCAGCAGCTTAAGGTTGAGCA,TAA,G,GGT,TAAG,12
2,AA,XV,RPA190,YOR341W,+,965969,965990,AATGCGGCTTAGAAGCAGGCA,TAG,A,AAG,TAGA,7
3,AA,VII,THI4,YGR144W,+,781367,781388,TTTGCTGCTTAGGCTTTTAAT,TAG,G,GCT,TAGG,8
4,AA,IV,FPR2,YDR519W,+,1480820,1480841,TCAGCCGCCTAGGTAAATTGG,TAG,G,GTA,TAGG,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
555,YY,XIV,unknown,YNL146C-A,-,351376,351397,TTATATTATTAAACCTATTTT,TAA,A,ACC,TAAA,11
556,YY,IV,unknown,YDR003W-A,+,454892,454913,ACATACTATTGATTTAGGTTT,TGA,T,TTT,TGAT,2
557,AA,IV,BSC2,YDR275W,+,1012947,1012968,AAAGCTGCTTAAATTGATAAT,TAA,A,ATT,TAAA,11
558,SS,XII,MDN1,YLR106C,-,348996,349017,GCATCCAGCTAAGTATATCGT,TAA,G,GTA,TAAG,12


In [240]:
#new_merged = new_merged[['Gene_name', 'Transcript_ID','Chromosome', 'Strand','Start','End','Sequence', 'amino_acid',
                         #'Stop Codon', '+4 nucleotide', '3UTR codon','4-stop signal', 'Score']]
new_merged = new_merged.rename(columns={'+4 nucleotide':'+4 Nucleotide'})

new_merged.to_csv('FilteredDuplicateEndings_Final.csv', index=False)

In [225]:
#make score lists of all QQ and YY endings
list_all_QQ = new_merged[new_merged['amino_acid'] != 'QQ']['Score'].tolist()
list_QQ = new_merged[new_merged['amino_acid'] == 'QQ']['Score'].tolist()

list_all_YY = new_merged[new_merged['amino_acid'] != 'YY']['Score'].tolist()
list_YY = new_merged[new_merged['amino_acid'] == 'YY']['Score'].tolist()

In [243]:
from scipy.stats import mannwhitneyu

stat, pvalue = mannwhitneyu(list_QQ, list_all_QQ, alternative='two-sided')
print(f"U statistic: {stat}, p-value: {pvalue}")

U statistic: 7794.5, p-value: 0.0005934297115955239


In [241]:
from scipy.stats import mannwhitneyu

stat, pvalue = mannwhitneyu(list_YY, list_all_YY, alternative='two-sided')
print(f"U statistic: {stat}, p-value: {pvalue}")

U statistic: 3033.0, p-value: 0.9635787044155879
